<a href="https://colab.research.google.com/github/mennawael05/Water-Stress-Detection-System/blob/main/Data%20Scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import ee
import numpy as np
import pandas as pd
from IPython.display import Image, display, HTML
from google.colab import files
import requests
from PIL import Image as PILImage
import io
import os

ee.Authenticate()
ee.Initialize(project='earth-engine-494017')


YEARS          = [2019, 2020, 2021, 2022, 2023]
NDVI_MIN_START = 0.25
MONOTONIC_TOL  = 0.08
BUFFER_METERS  = 80_000
NUM_PIXELS     = 10
ROI_BUFFER_M   = 500
CLOUD_MAX      = 20

OUTPUT_NPY     = 'water_stress_tensors.npy'
OUTPUT_LABELS  = 'water_stress_labels.npy'
OUTPUT_CSV     = 'water_stress_metadata.csv'

# (n_points, n_years, n_seasons, n_imgs, n_bands)
# n_years=5, n_seasons=4, n_imgs=4, n_bands=7
N_YEARS   = 5
N_SEASONS = 4
N_IMGS    = 4
N_BANDS   = 7   # NDVI, BSI, NDWI, NDMI, LSWI, EVI, SAVI
BAND_NAMES = ['NDVI', 'BSI', 'NDWI', 'NDMI', 'LSWI', 'EVI', 'SAVI']

SEASONAL_MONTHS = [
    {"name": "MAR", "start": "03-01", "end": "03-31"},
    {"name": "JUN", "start": "06-01", "end": "06-30"},
    {"name": "SEP", "start": "09-01", "end": "09-30"},
    {"name": "DEC", "start": "12-01", "end": "12-31"},
]

FALLBACK = {
    "MAR": [("02-01","02-28"), ("04-01","04-30")],
    "JUN": [("05-01","05-31"), ("07-01","07-31")],
    "SEP": [("08-01","08-31"), ("10-01","10-31")],
    "DEC": [("11-01","11-30"), ("01-01","01-31")],
}

'''
{"id": "usa_california", "lon": -119.500, "lat": 36.500, "scenario": "B", "severity": 3},
{"id": "thailand_chao",     "lon": 100.00, "lat": 15.50,  "scenario": "A", "severity": 2},
{"id": "tunisia_medjerda",  "lon": 10.00,   "lat": 36.50,  "scenario": "A", "severity": 3},
'''


REGIONS = [
    {"id": "iraq_babil",      "lon": 44.50, "lat": 32.50, "scenario": "A", "severity": 3},
    {"id": "iraq_diyala",     "lon": 44.92, "lat": 33.97, "scenario": "A", "severity": 2},
    {"id": "iraq_najaf",      "lon": 44.30, "lat": 31.80, "scenario": "A", "severity": 3},
    {"id": "pakistan_indus",  "lon": 68.50, "lat": 27.50, "scenario": "A", "severity": 3},
    {"id": "myanmar_chindwin", "lon": 95.70, "lat": 22.50, "scenario": "A", "severity": 2},
    {"id": "laos_mekong_mid",  "lon":102.50, "lat": 17.50, "scenario": "A", "severity": 2},
    {"id": "vietnam_mekong_up","lon":105.00, "lat": 11.50, "scenario": "A", "severity": 3},

    {"id": "india_punjab_gw", "lon": 75.50, "lat": 30.50, "scenario": "B", "severity": 3},
    {"id": "india_haryana",   "lon": 76.50, "lat": 29.50, "scenario": "B", "severity": 3},

    {"id": "sahel_region",    "lon": 2.0,   "lat": 14.0,  "scenario": "C", "severity": 3},
    {"id": "gobi_desert",     "lon": 105.0, "lat": 44.0,  "scenario": "C", "severity": 3},
    {"id": "mongolia_inner",  "lon": 103.8, "lat": 46.8,  "scenario": "C", "severity": 2},
    {"id": "inner_mongolia_2","lon": 113.0, "lat": 44.0,  "scenario": "C", "severity": 2},
    {"id": "mu_us_desert",    "lon": 109.0, "lat": 38.5,  "scenario": "C", "severity": 3},
    {"id": "argentina_degrade","lon": -65.0, "lat": -38.0, "scenario": "C", "severity": 2},
    {"id": "mexico_desert",   "lon": -102.0,"lat": 23.0,  "scenario": "C", "severity": 2},
    {"id": "latam_andes",     "lon": -72.0, "lat": -15.0, "scenario": "C", "severity": 3},

]

CONFIRMED_POINTS = [
    {"id": "iraq_confirmed_1", "lat": 32.619, "lon": 44.814,
     "scenario": "A", "severity": 3},
]

def add_indices(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    bsi  = image.expression(
        '((B11 + B4) - (B8 + B2)) / ((B11 + B4) + (B8 + B2))',
        {'B11': image.select('B11'), 'B4': image.select('B4'),
         'B8':  image.select('B8'),  'B2': image.select('B2')}
    ).rename('BSI')
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    lswi = image.normalizedDifference(['B8', 'B12']).rename('LSWI')
    evi  = image.expression(
        '2.5 * ((B8 - B4) / (B8 + 6 * B4 - 7.5 * B2 + 1))',
        {'B8': image.select('B8'), 'B4': image.select('B4'), 'B2': image.select('B2')}
    ).rename('EVI')
    savi = image.expression(
        '((B8 - B4) / (B8 + B4 + 0.5)) * 1.5',
        {'B8': image.select('B8'), 'B4': image.select('B4')}
    ).rename('SAVI')
    return image.addBands([ndvi, bsi, ndwi, ndmi, lswi, evi, savi])

def get_cropland_points(center_lon, center_lat, n=NUM_PIXELS):
    region    = ee.Geometry.Point([center_lon, center_lat]).buffer(BUFFER_METERS)
    landcover = ee.Image('MODIS/006/MCD12Q1/2019_01_01').select('LC_Type1')
    cropland  = landcover.eq(12).Or(landcover.eq(14))
    water     = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    no_water  = water.lt(50).Or(water.unmask(0).eq(0))
    no_urban  = landcover.neq(13)
    mask      = cropland.And(no_water).And(no_urban)
    return mask.selfMask().sample(
        region=region, scale=500, numPixels=n, seed=42, geometries=True
    )

def get_4_images(roi, year, season_name, s_start, s_end):
    months_to_try = [(s_start, s_end)] + FALLBACK.get(season_name, [])
    for m_start, m_end in months_to_try:
        col = (
            ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterBounds(roi)
            .filterDate(f'{year}-{m_start}', f'{year}-{m_end}')
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', CLOUD_MAX))
            .map(add_indices)
            .sort('system:time_start')
        )
        count = col.size().getInfo()
        if count == 0:
            continue
        imgs_info = col.toList(min(count, 20)).getInfo()
        if not imgs_info:
            continue

        total = len(imgs_info)
        if   total == 1: indices = [0, 0, 0, 0]
        elif total == 2: indices = [0, 0, 1, 1]
        elif total == 3: indices = [0, 1, 2, 2]
        else:            indices = [0, total//3, (2*total)//3, total-1]

        selected = []
        seen_dates = set()
        for idx in indices:
            info     = imgs_info[idx]
            date_ms  = info['properties'].get('system:time_start', 0)
            date_str = pd.Timestamp(date_ms, unit='ms').strftime('%Y-%m-%d')
            cloud    = info['properties'].get('CLOUDY_PIXEL_PERCENTAGE', 0)
            img_ee   = add_indices(ee.Image(info['id']))
            if date_str in seen_dates:
                for alt in imgs_info:
                    alt_date = pd.Timestamp(
                        alt['properties'].get('system:time_start', 0), unit='ms'
                    ).strftime('%Y-%m-%d')
                    if alt_date not in seen_dates:
                        date_str = alt_date
                        cloud    = alt['properties'].get('CLOUDY_PIXEL_PERCENTAGE', 0)
                        img_ee   = add_indices(ee.Image(alt['id']))
                        break
            seen_dates.add(date_str)
            selected.append({'img': img_ee, 'date': date_str,
                             'cloud': round(cloud, 1), 'month_used': m_start[:2]})

        while len(selected) < 4:
            selected.append(selected[-1])
        return selected[:4]
    return []

def get_stats(image, roi):
    result = {}
    for band in BAND_NAMES:
        stats = image.select(band).reduceRegion(
            reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
            geometry=roi, scale=10
        )
        result[f'{band}_mean'] = stats.get(f'{band}_mean').getInfo()
        result[f'{band}_std']  = stats.get(f'{band}_stdDev').getInfo()
    return result

def get_thumbnail_url(image, roi):
    return image.select(['B4', 'B3', 'B2']).getThumbURL({
        'min': 0, 'max': 2800, 'gamma': 1.5,
        'dimensions': 400,
        'region': roi.buffer(3000),
        'format': 'png'
    })


def compute_slope(values):
    valid = [(y, v) for y, v in zip(YEARS, values) if v is not None]
    if len(valid) < 3:
        return None
    xs = np.array([p[0] for p in valid], dtype=float)
    ys = np.array([p[1] for p in valid], dtype=float)
    xs -= xs.mean()
    return round(float(np.dot(xs, ys) / np.dot(xs, xs)), 6)

def is_monotonic(vals, tol=MONOTONIC_TOL):
    v = [x for x in vals if x is not None]
    if len(v) < 3 or v[-1] >= v[0]:
        return False
    if max(v) > v[0] + tol:
        return False
    ups = [v[i]-v[i-1] for i in range(1, len(v)) if v[i] > v[i-1]]
    return not (len(ups) > 1 or (ups and ups[0] > tol))

def interpret(ndvi, bsi):
    ndvi = ndvi or 0; bsi = bsi or 0
    if   ndvi > 0.35 and bsi < 0.0: return '🟢 خضراء'
    elif ndvi > 0.35:                return '🟡 خضراء/جافة'
    elif ndvi > 0.15 and bsi < 0.0: return '🟡 بتتدهور'
    elif ndvi > 0.15:                return '🟠 تدهور واضح'
    else:                            return '🔴 جفاف'

def interpret_water(ndwi, ndmi, lswi):
    """تفسير مؤشرات المياه الثلاثة معاً"""
    parts = []

    # NDWI: مياه سطحية
    if ndwi is not None:
        if   ndwi > 0.2:  parts.append('💧 مياه سطحية موجودة')
        elif ndwi > 0.0:  parts.append('🔵 رطوبة سطحية خفيفة')
        else:              parts.append('🟤 لا مياه سطحية')

    # NDMI: رطوبة التربة
    if ndmi is not None:
        if   ndmi > 0.2:  parts.append('🌱 تربة رطبة')
        elif ndmi > 0.0:  parts.append('🌾 رطوبة تربة متوسطة')
        else:              parts.append('🏜️ تربة جافة')

    # LSWI: إجهاد مائي للنبات
    if lswi is not None:
        if   lswi > 0.2:  parts.append('✅ لا إجهاد مائي')
        elif lswi > 0.0:  parts.append('⚠️ إجهاد مائي خفيف')
        else:              parts.append('🚨 إجهاد مائي حاد')

    return ' | '.join(parts)

def auto_label(meta):
    if meta['scenario'] == 'E':
        return 0
    ndvi_first = meta.get('ndvi_mean_MAR_2019') or meta.get('ndvi_mean_MAR_2020') or 0
    ndvi_last  = meta.get('ndvi_mean_MAR_2023') or 0
    ndvi_drop  = ndvi_first - ndvi_last
    lswi  = meta.get('lswi_mean_JUN_2023', 0) or 0
    ndmi  = meta.get('ndmi_mean_JUN_2023', 0) or 0
    bsi   = meta.get('bsi_mean_JUN_2023',  0) or 0
    slope = meta.get('ndvi_slope_MAR', 0)     or 0
    score = 0
    if ndvi_drop > 0.10: score += 2
    if ndvi_drop > 0.05: score += 1
    if lswi < 0.0:       score += 2
    if ndmi < -0.05:     score += 2
    if bsi  > 0.10:      score += 1
    if slope < -0.02:    score += 2
    return 1 if score >= 4 else 0

def water_stress_score(meta):
    lswi  = meta.get('lswi_mean_JUN_2023', 0) or 0
    ndmi  = meta.get('ndmi_mean_JUN_2023', 0) or 0
    ndwi  = meta.get('ndwi_mean_JUN_2023', 0) or 0
    slope = meta.get('ndvi_slope_MAR', 0)     or 0
    bsi   = meta.get('bsi_mean_JUN_2023',  0) or 0
    s1 = max(0, min(1, (-lswi  + 0.3) / 0.6))
    s2 = max(0, min(1, (-ndmi  + 0.2) / 0.5))
    s3 = max(0, min(1, (-ndwi  + 0.1) / 0.5))
    s4 = max(0, min(1, (-slope + 0.05)/ 0.1))
    s5 = max(0, min(1, (bsi    + 0.0) / 0.3))
    return round(0.30*s1 + 0.25*s2 + 0.15*s3 + 0.20*s4 + 0.10*s5, 3)


def process_point(pt_id, pt_lon, pt_lat, scenario, severity,
                  all_tensors, all_labels, all_metadata):
    """
    بترجع:
    - tensor shape: (N_YEARS, N_SEASONS, N_IMGS, N_BANDS)
    - label: int
    - metadata: dict (للـ CSV)
    """
    roi = ee.Geometry.Point([pt_lon, pt_lat]).buffer(ROI_BUFFER_M)

    tensor = np.full((N_YEARS, N_SEASONS, N_IMGS, N_BANDS), np.nan, dtype=np.float32)

    meta = {
        'pt_id':    pt_id,
        'lat':      round(pt_lat, 4),
        'lon':      round(pt_lon, 4),
        'scenario': scenario,
        'severity': severity,
    }

    # ndvi series لكل شهر للـ slope
    ndvi_series = {s['name']: [] for s in SEASONAL_MONTHS}

    display(HTML(f'''
    <div style="background:#1a1a2e;color:#e0e0e0;padding:12px;border-radius:8px;
                border-left:4px solid #4fc3f7;margin:16px 0">
        <b style="color:#4fc3f7">📍 {pt_id}</b> &nbsp;|&nbsp;
        ({pt_lat:.3f}, {pt_lon:.3f}) &nbsp;|&nbsp;
        Scenario=<b style="color:#ff9800">{scenario}</b> &nbsp;|&nbsp;
        Severity=<b style="color:#ef5350">{severity}</b>
    </div>
    '''))

    for y_idx, year in enumerate(YEARS):
        display(HTML(f'<h4 style="color:#81d4fa;margin:12px 0 4px">📅 {year}</h4>'))

        for s_idx, season in enumerate(SEASONAL_MONTHS):
            sname  = season['name']
            s_start= season['start']
            s_end  = season['end']

            imgs = get_4_images(roi, year, sname, s_start, s_end)

            if not imgs:
                ndvi_series[sname].append(None)
                display(HTML(f'<span style="color:#ef5350">⚠️ {sname} — مفيش صور</span>'))
                continue

            display(HTML(f'<b style="color:#a5d6a7">{sname}</b>'))

            year_ndvi_img1 = None
            for i_idx, img_data in enumerate(imgs):
                stats    = get_stats(img_data['img'], roi)
                ndvi_val = stats.get('NDVI_mean')
                bsi_val  = stats.get('BSI_mean')
                ndwi_val = stats.get('NDWI_mean')
                ndmi_val = stats.get('NDMI_mean')
                lswi_val = stats.get('LSWI_mean')

                if i_idx == 0:
                    year_ndvi_img1 = ndvi_val

                for b_idx, band in enumerate(BAND_NAMES):
                    val = stats.get(f'{band}_mean')
                    if val is not None:
                        tensor[y_idx, s_idx, i_idx, b_idx] = val

                for band in BAND_NAMES:
                    key = f'{band.lower()}_mean_{sname}_{year}'
                    meta[key] = stats.get(f'{band}_mean')

                ndvi_str = f'{ndvi_val:.3f}' if ndvi_val is not None else 'N/A'
                bsi_str  = f'{bsi_val:.3f}'  if bsi_val  is not None else 'N/A'
                fb_note  = f'[fallback:{img_data["month_used"]}]' \
                           if img_data['month_used'] != s_start[:2] else ''
                print(f'  [{i_idx+1}] {img_data["date"]} ☁️{img_data["cloud"]}% | '
                      f'NDVI={ndvi_str} BSI={bsi_str} | '
                      f'{interpret(ndvi_val, bsi_val)} {fb_note}')
                print(f'       {interpret_water(ndwi_val, ndmi_val, lswi_val)}')

                url = get_thumbnail_url(img_data['img'], roi)
                display(Image(url=url, width=260))

            ndvi_series[sname].append(year_ndvi_img1)

    for sname in [s['name'] for s in SEASONAL_MONTHS]:
        slope = compute_slope(ndvi_series[sname])
        meta[f'ndvi_slope_{sname}']   = slope
        meta[f'is_monotonic_{sname}'] = int(is_monotonic(ndvi_series[sname]))
        vals = [v for v in ndvi_series[sname] if v is not None]
        meta[f'ndvi_drop_{sname}']    = round(vals[0] - vals[-1], 4) if len(vals) >= 2 else None

    label       = auto_label(meta)
    score       = water_stress_score(meta)
    meta['label_binary']       = label
    meta['water_stress_score'] = score
    meta['severity_label']     = severity

    all_tensors.append(tensor)
    all_labels.append(label)
    all_metadata.append(meta)

    display(HTML(f'''
    <div style="background:#1b5e20;color:#c8e6c9;padding:8px 12px;
                border-radius:6px;margin:8px 0">
        ✅ label=<b>{label}</b> | score=<b>{score}</b> |
        slope_MAR=<b>{meta.get("ndvi_slope_MAR")}</b>
    </div>
    '''))


all_tensors  = []
all_labels   = []
all_metadata = []

display(HTML('<h2 style="color:#ff9800">📌 CONFIRMED POINTS</h2>'))
for pt in CONFIRMED_POINTS:
    process_point(pt['id'], pt['lon'], pt['lat'],
                  pt['scenario'], pt['severity'],
                  all_tensors, all_labels, all_metadata)

display(HTML('<h2 style="color:#ff9800">🌍 REGIONS LOOP</h2>'))
for region in REGIONS:
    rid      = region['id']
    scenario = region['scenario']
    severity = region['severity']

    display(HTML(f'<h3 style="color:#b0bec5">🔍 {rid} | {scenario} | sev={severity}</h3>'))

    points   = get_cropland_points(region['lon'], region['lat'])
    pts_list = points.toList(NUM_PIXELS).getInfo()

    if not pts_list:
        print(f'  ⚠️ مفيش أراضي زراعية — skip')
        continue

    print(f'  📍 {len(pts_list)} نقطة زراعية')

    for i, pt in enumerate(pts_list):
        coords         = pt['geometry']['coordinates']
        pt_lon, pt_lat = coords[0], coords[1]
        process_point(f'{rid}_{i+1}', pt_lon, pt_lat,
                      scenario, severity,
                      all_tensors, all_labels, all_metadata)

#saving
display(HTML('<h2 style="color:#ff9800">💾 حفظ الملفات</h2>'))

tensors_array = np.stack(all_tensors, axis=0)
labels_array  = np.array(all_labels,  dtype=np.int8)

np.save(OUTPUT_NPY,    tensors_array)
np.save(OUTPUT_LABELS, labels_array)

print(f'✅ Tensors: {tensors_array.shape}  → {OUTPUT_NPY}')
print(f'✅ Labels:  {labels_array.shape}   → {OUTPUT_LABELS}')
print(f'   📦 حجم الـ tensor: {tensors_array.nbytes / 1e9:.2f} GB')

df = pd.DataFrame(all_metadata)

meta_cols  = ['pt_id','lat','lon','scenario','severity']
label_cols = ['label_binary','water_stress_score','severity_label']
slope_cols = [c for c in df.columns if 'slope' in c or 'drop' in c or 'monotonic' in c]
feat_cols  = [c for c in df.columns
              if c not in meta_cols + label_cols + slope_cols]

df = df[meta_cols + feat_cols + slope_cols + label_cols]
df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print(f'✅ CSV: {OUTPUT_CSV} | {len(df)} صف | {len(df.columns)} عمود')

#summary
display(HTML('<h3 style="color:#4fc3f7">📊 ملخص الداتاست</h3>'))
print(f'عدد النقاط: {len(df)}')
print(f'\n🏷️ Labels:')
print(df['label_binary'].value_counts())
print(f'\n🗺️ Scenarios:')
print(df['scenario'].value_counts())
print(f'\n💧 Water Stress Score:')
print(df['water_stress_score'].describe())

display(HTML('<h3 style="color:#ff9800">⬇️ تنزيل الملفات</h3>'))
files.download(OUTPUT_NPY)
files.download(OUTPUT_LABELS)
files.download(OUTPUT_CSV)

  [1] 2019-03-05 ☁️1.6% | NDVI=0.532 BSI=-0.146 | 🟢 خضراء 
       🟤 لا مياه سطحية | 🌱 تربة رطبة | ✅ لا إجهاد مائي


  [2] 2019-03-10 ☁️0.1% | NDVI=0.580 BSI=-0.167 | 🟢 خضراء 
       🟤 لا مياه سطحية | 🌱 تربة رطبة | ✅ لا إجهاد مائي


  [3] 2019-03-20 ☁️11.2% | NDVI=0.563 BSI=-0.161 | 🟢 خضراء 
       🟤 لا مياه سطحية | 🌱 تربة رطبة | ✅ لا إجهاد مائي


  [4] 2019-03-25 ☁️6.5% | NDVI=0.603 BSI=-0.200 | 🟢 خضراء 
       🟤 لا مياه سطحية | 🌱 تربة رطبة | ✅ لا إجهاد مائي


  [1] 2019-06-03 ☁️0.0% | NDVI=0.153 BSI=0.123 | 🟠 تدهور واضح 
       🟤 لا مياه سطحية | 🏜️ تربة جافة | ⚠️ إجهاد مائي خفيف


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>